# **Random_forest_regressor**

## **Các bước thực hiện:** 

#### **1. Khai báo thư viện, đọc data từ file csv**

In [2]:
# Đọc file
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error

filepath = '../data/processed/'
df_test = pd.read_csv(filepath + 'test_data_final.csv')
df_train = pd.read_csv(filepath + 'train_data_final.csv')

In [3]:
df_train.head()

,price,original_price,discount_rate,quantity_sold,rating_average,review_count,is_return_policy,is_freeship_xtra,is_authentic,image_count,...,category_root_name_Điện Thoại - Máy Tính Bảng,category_root_name_Điện Tử - Điện Lạnh,category_root_name_Đồ chơi - Mẹ & Bé,China,Japan,South Korea,Thailand,USA,Vietnam,Others
0,0.445108,0.438575,0.000000,3.828641,0.74,0.331717,1.0,1.0,1.0,0.411765,...,0,0,0,1,1,0,0,0,0,0.00
1,0.313109,0.343930,0.500000,3.828641,1.00,0.367314,1.0,0.0,0.0,0.000000,...,0,0,0,0,0,0,0,0,0,0.25
2,0.156620,0.148556,0.000000,0.000000,0.00,0.000000,0.0,1.0,1.0,0.176471,...,0,0,0,0,0,0,0,0,1,0.00
3,0.263237,0.330742,0.846154,3.218876,0.86,0.310416,1.0,1.0,1.0,0.294118,...,0,0,0,0,0,0,0,0,0,0.25
4,0.396891,0.390102,0.000000,1.945910,0.80,0.175253,1.0,1.0,1.0,0.235294,...,0,0,0,1,0,0,1,0,0,0.00


In [4]:
df_test.head()

,price,original_price,discount_rate,quantity_sold,rating_average,review_count,is_return_policy,is_freeship_xtra,is_authentic,image_count,...,category_root_name_Điện Thoại - Máy Tính Bảng,category_root_name_Điện Tử - Điện Lạnh,category_root_name_Đồ chơi - Mẹ & Bé,China,Japan,South Korea,Thailand,USA,Vietnam,Others
0,0.069133,0.060604,0.000000,3.218876,1.0,0.175253,1.0,1.0,1.0,0.117647,...,0,0,0,0,0,0,0,0,1,0.00
1,0.470614,0.464217,0.000000,0.000000,0.0,0.000000,1.0,1.0,1.0,0.411765,...,0,0,0,0,0,0,0,0,1,0.00
2,0.000000,0.000000,0.000000,1.791759,0.0,0.000000,1.0,1.0,1.0,0.000000,...,0,0,0,0,0,0,0,0,1,0.00
3,0.443963,0.444183,0.096154,3.465736,0.9,0.382518,1.0,1.0,1.0,0.588235,...,0,0,0,0,0,0,0,0,0,0.25
4,0.659291,0.653896,0.000000,0.000000,0.0,0.000000,1.0,1.0,1.0,0.352941,...,0,0,0,0,0,0,0,1,0,0.00


In [5]:
# Tách X, y cho tập train
X_train = df_train.drop(columns=['quantity_sold','review_to_sold_ratio'])
y_train = df_train['quantity_sold']

# Tách X, y cho tập train
X_test = df_test.drop(columns=['quantity_sold','review_to_sold_ratio'])
y_test = df_test['quantity_sold']

#### **2. Huấn luyện mô hình Decision Tree Regressor**

In [6]:
# Huấn luyện mô hình cây (Decision Tree Regressor)
rf = RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)

#### **3. Tinh chỉnh mô hình (Finetune)**

**3.1. Khai báo grid siêu tham số và chạy GridSearchCV**

In [7]:
param_grid = {
    "n_estimators": [200, 500],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt", "log2"]
}

grid_rf = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=3,
    scoring="r2",
    n_jobs=-1,
    verbose=1
)

grid_rf.fit(X_train, y_train)

Fitting 3 folds for each of 48 candidates, totalling 144 fits


GridSearchCV(cv=3, estimator=RandomForestRegressor(n_jobs=-1, random_state=42),
             n_jobs=-1,
             param_grid={'max_depth': [None, 10, 20],
                         'max_features': ['sqrt', 'log2'],
                         'min_samples_leaf': [1, 2],
                         'min_samples_split': [2, 5],
                         'n_estimators': [200, 500]},
             scoring='r2', verbose=1)

**3.2. Chọn mô hình tốt nhất**

In [8]:
# Lấy mô hình tốt nhất
print("Best RF params:")
print(grid_rf.best_params_)

rf_best = grid_rf.best_estimator_

Best RF params:
{'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 500}


### **4. Đánh giá trên validation/test**

In [9]:
y_test_pred = rf_best.predict(X_test)

r2 = r2_score(y_test, y_test_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

print(f"RF R2 (test): {r2:.4f}")
print(f"RF RMSE (test): {rmse:.4f}")

RF R2 (test): 0.9026
RF RMSE (test): 0.7140


#### **5. Feature Importance (model đã tune)**

In [10]:
rf_feature_importances = pd.Series(
    rf_best.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print("Random Forest Feature Importances:")
print(rf_feature_importances.to_string())

Random Forest Feature Importances:
review_count                                           0.268273
reputation_score                                       0.228847
rating_average                                         0.139222
store_review_count                                     0.063246
total_follower                                         0.045372
shop_potential                                         0.045286
price                                                  0.025135
original_price                                         0.024679
price_vs_category                                      0.019496
is_official                                            0.014646
name_length                                            0.011607
trust_level                                            0.009955
name_word_count                                        0.009502
discount_amount                                        0.008701
total_visuals                                          0.007884
image

### **6. Lưu kết quả dự đoán**

In [11]:
# Tạo dataframe kết quả
result_df = pd.DataFrame({
    'quantity_sold_ground_truth': y_test.values,
    'quantity_sold_predicted': y_test_pred
})

result_df.to_csv(
    '../modeling/rf_predictions.csv',
    index=False
)